# Lab 05-02: Build an ADK Agent With Tools, Callbacks, and a Schema

**Book chapter:** Chapter 05: Models and Building Agents With ADK  
**Estimated time:** 30 minutes

## Overview

Cymbal Retail's order assistant must apply the returns policy exactly, refuse payment
questions without involving the model, and hand a structured record of each return
decision to the returns system. In this lab, you build the agent with the Agent
Development Kit (ADK) in code, with two function tools, a before-tool callback, a
before-model callback, and `output_key`, and you add a formatter agent that uses
`output_schema`.

## Learning Objectives

By the end of this lab, you will be able to:

- Create ADK function tools that read and write session state
- Enforce rules with before-tool and before-model callbacks
- Run an agent programmatically with a Runner and a session service
- Produce structured output with output_schema and pass data with output_key

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** A few cents for Gemini requests. No resources are created.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| ADK feature | Purpose in this lab |
|---|---|
| **Function tool** | Looks up orders and applies the returns policy in code |
| **before_tool_callback** | Validates tool arguments before a tool runs |
| **before_model_callback** | Answers some requests without calling the model |
| **output_key** | Saves the agent's final answer to session state |
| **output_schema** | Forces the final answer to match a Pydantic model |

## Setup

In [ ]:
%pip install --quiet --upgrade google-adk

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The lab code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Configure ADK and define the tools

ADK reads its connection settings from environment variables, so the cell sets
`GOOGLE_GENAI_USE_VERTEXAI` and the location. The two tools are plain Python functions;
ADK turns their names, type hints, and docstrings into tool declarations.

Notice that `check_return_eligibility` applies the returns policy in **code**. The model
decides when to call it and how to explain the result, but it never decides the rule.

In [ ]:
import asyncio
import os
from concurrent.futures import ThreadPoolExecutor

from google.adk.agents import Agent
from google.adk.models import LlmResponse
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import ToolContext
from google.genai import types
from pydantic import BaseModel

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")

ORDERS = {
    "10421": {"item": "Trailhead boots", "shipping": "in transit",
              "days_since_delivery": None},
    "10422": {"item": "Rain jacket", "shipping": "delivered",
              "days_since_delivery": 12},
    "10423": {"item": "Hiking boots", "shipping": "delivered",
              "days_since_delivery": 45},
}


def get_order_status(order_id: str, tool_context: ToolContext) -> dict:
    """Returns the item, shipping status, and delivery age of an order.

    Args:
        order_id: The five-digit Cymbal Retail order number.
    """
    order = ORDERS.get(order_id)
    if order is None:
        return {"found": False, "message": f"Order {order_id} not found."}
    tool_context.state["last_order_id"] = order_id
    return {"found": True, "order_id": order_id, **order}


def check_return_eligibility(order_id: str, worn_outdoors: bool,
                             defective: bool, gold_member: bool) -> dict:
    """Applies the Cymbal Retail returns policy to a delivered order.

    Args:
        order_id: The five-digit order number.
        worn_outdoors: Whether the item has been worn outdoors.
        defective: Whether the item is defective.
        gold_member: Whether the shopper is a Gold-tier member.
    """
    order = ORDERS.get(order_id)
    if order is None or order["days_since_delivery"] is None:
        return {"eligible": False, "rule": "Not delivered yet"}
    days = order["days_since_delivery"]
    if defective:
        eligible, rule = days <= 365, "Defective items: 12 months"
    elif worn_outdoors:
        eligible, rule = False, "Items worn outdoors cannot be returned"
    else:
        limit = 60 if gold_member else 30
        eligible, rule = days <= limit, f"Unworn items: {limit} days"
    return {"eligible": eligible, "days_since_delivery": days, "rule": rule}

## Step 2: Add callbacks that enforce rules in code

Callbacks enforce rules deterministically:

- `validate_order_id` runs **before each tool call**. Returning a dictionary skips the
  tool and uses the dictionary as its result.
- `block_payment_questions` runs **before each model call**. Returning an `LlmResponse`
  skips the model entirely.

Returning `None` lets the normal processing continue.

In [ ]:
def validate_order_id(tool, args, tool_context):
    """Before-tool callback: blocks calls with a malformed order number."""
    order_id = str(args.get("order_id", ""))
    if not (order_id.isdigit() and len(order_id) == 5):
        return {"found": False, "message": "Order numbers have 5 digits."}
    return None  # None lets the tool run.


def block_payment_questions(callback_context, llm_request):
    """Before-model callback: answers payment questions without a model."""
    text = " ".join(p.text or "" for p in llm_request.contents[-1].parts)
    if any(word in text.lower() for word in ("credit card", "password")):
        return LlmResponse(content=types.Content(role="model", parts=[
            types.Part(text="For your security, I can't help with card "
                            "details or passwords. Please use your "
                            "account page.")]))
    return None  # None lets the request go to the model.

## Step 3: Create the agent and a helper that shows each event

The agent combines the tools and callbacks. `output_key="last_answer"` saves its final
answer to session state. The `chat` helper runs the agent with a `Runner` and an
in-memory session service and prints each tool call (`->`), tool result (`<-`), and the
final answer. `state_of` reads a session's state.

In [ ]:
assistant = Agent(
    name="order_assistant",
    model=MODEL,
    description="Answers Cymbal Retail order and return questions.",
    instruction="You help Cymbal Retail shoppers with orders and returns. "
                "Always use the tools and never guess. Before checking a "
                "return, make sure you know whether the item was worn "
                "outdoors, whether it is defective, and whether the "
                "shopper is a Gold member. Keep answers short.",
    tools=[get_order_status, check_return_eligibility],
    before_tool_callback=validate_order_id,
    before_model_callback=block_payment_questions,
    output_key="last_answer",
)
sessions = InMemorySessionService()


def chat(agent, session_id, text, user_id="shopper-1", state=None):
    """Sends one message and prints the tool calls and the final answer."""
    runner = Runner(app_name="cymbal", agent=agent,
                    session_service=sessions, auto_create_session=True)
    print(f"\nUser: {text}")
    message = types.Content(role="user", parts=[types.Part(text=text)])
    for event in runner.run(user_id=user_id, session_id=session_id,
                            new_message=message, state_delta=state):
        for call in event.get_function_calls():
            print(f"  -> {call.name}({dict(call.args)})")
        for reply in event.get_function_responses():
            print(f"  <- {reply.response}")
        if event.is_final_response() and event.content and \
                event.content.parts:
            print("Agent:", event.content.parts[0].text)


def state_of(session_id, user_id="shopper-1"):
    """Returns a copy of a session's state."""
    lookup = sessions.get_session(app_name="cymbal", user_id=user_id,
                                  session_id=session_id)
    with ThreadPoolExecutor(1) as pool:  # Works in scripts and notebooks.
        return dict(pool.submit(asyncio.run, lookup).result().state)

## Step 4: Test the tools and the callbacks

Test the tools and both callbacks.

In [ ]:
chat(assistant, "s1", "Where is order 10421?")
chat(assistant, "s1", "And what about order 12AB?")
chat(assistant, "s1", "Can I pay with a different credit card?")
chat(assistant, "s2", "I want to return the boots from order 10423. "
     "They are unworn and not defective, and I'm a Gold member.")
print("\nState of session s2:", state_of("s2"))

**Expected results:**

| Request | Expected behavior |
|---|---|
| Where is order 10421? | `get_order_status` reports the boots are in transit |
| What about order 12AB? | The before-tool callback blocks the malformed number |
| Credit card question | The before-model callback answers; the model is not used |
| Return order 10423 | `check_return_eligibility` applies the 60-day Gold rule and returns eligible |

## Step 5: Turn the answer into structured output with a second agent

An agent with `output_schema` returns JSON that must match a Pydantic model. Because
such an agent does not call tools, a separate formatter agent converts the assessment,
passed to it through state, into a `ReturnDecision` record.

In [ ]:
class ReturnDecision(BaseModel):
    order_id: str
    eligible: bool
    rule: str
    next_step: str


formatter = Agent(
    name="return_formatter",
    model=MODEL,
    instruction="Convert this return assessment into the required fields: "
                "{last_answer}",
    output_schema=ReturnDecision,
    output_key="return_decision",
)
assessment = state_of("s2")["last_answer"]
chat(formatter, "s3", "Create the return decision record.",
     state={"last_answer": assessment})
print("\nStructured record:", state_of("s3")["return_decision"])

**Expected result:** a dictionary with `order_id`, `eligible`, `rule`, and `next_step`.

## Step 6: Experiment

- In Step 4, change the last request to describe a standard member instead of a Gold
  member, rerun Step 4, and confirm that the 30-day rule makes the return ineligible.
- In Step 5, add `refund_method: str` to `ReturnDecision`, rerun Step 5, and notice that the
  record now includes it even though the instruction never mentions it.

## Check Your Understanding

**1. Why does the returns policy live in a tool instead of in the instruction?**

<details><summary>Show answer</summary>

Code applies the rule the same way every time. The model only decides when to call the tool and how to explain the result.

</details>

**2. What happens when a before-model callback returns an LlmResponse?**

<details><summary>Show answer</summary>

ADK uses that response as the model's reply and skips the model call.

</details>

**3. Why is the structured record produced by a second agent?**

<details><summary>Show answer</summary>

An agent with output_schema returns structured data instead of calling tools, so reasoning with tools and formatting are separated into two agents.

</details>

## Summary

You built an ADK agent with tools that read and write state, callbacks that enforce rules
in code, and a formatter that returns schema-validated JSON. This lab created no resources,
so there is nothing to clean up.